# Commodity Price Forecaster — Model Development
Prototype ARIMA, Prophet, and weather-feature models before deployment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, '..')

from src.data_loader import fetch_commodity_prices, fetch_weather_data, simulate_weather_data
from src.forecaster import fit_best_arima, fit_prophet, decompose_series, compute_metrics
from src.utils import compute_risk_scores

plt.style.use('dark_background')
print('Libraries loaded!')

## 1. Load Commodity Data

In [ ]:
prices = fetch_commodity_prices('CL=F', '2020-01-01', '2024-12-31')
weekly = prices.resample('W').last().dropna()
print(f'Loaded {len(weekly)} weekly observations for WTI Crude Oil')
weekly.tail()

## 2. Auto-ARIMA Forecast

In [ ]:
result = fit_best_arima(weekly, horizon=12, alpha=0.05)
print('ARIMA order:', result['order'])
print('Metrics:', result['metrics'])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(weekly.index[-52:], weekly.values[-52:], label='Historical', color='#636EFA')
fc = result['forecast_df']
ax.plot(fc.index, fc['forecast'], '--', color='red', label='ARIMA Forecast')
ax.fill_between(fc.index, fc['lower'], fc['upper'], alpha=0.2, color='red')
ax.legend(); ax.set_title('WTI Crude Oil - ARIMA Forecast')
plt.tight_layout(); plt.show()

## 3. Prophet Forecast

In [ ]:
p_result = fit_prophet(weekly, horizon=12)
if p_result:
    print('Prophet Metrics:', p_result['metrics'])
    fc_p = p_result['forecast_df']
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(weekly.index[-52:], weekly.values[-52:], label='Historical', color='#636EFA')
    ax.plot(fc_p.index, fc_p['forecast'], '--', color='green', label='Prophet Forecast')
    ax.fill_between(fc_p.index, fc_p['lower'], fc_p['upper'], alpha=0.2, color='green')
    ax.legend(); ax.set_title('WTI Crude Oil - Prophet Forecast')
    plt.tight_layout(); plt.show()
else:
    print('Prophet not installed. Install: pip install prophet')

## 4. STL Decomposition

In [ ]:
decomp = decompose_series(weekly)
if decomp:
    fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
    for ax, data, label, color in zip(
        axes,
        [decomp.observed, decomp.trend, decomp.seasonal, decomp.resid],
        ['Observed', 'Trend', 'Seasonal', 'Residual'],
        ['#636EFA', 'red', 'green', 'purple']
    ):
        ax.plot(pd.Series(data).dropna().index,
                pd.Series(data).dropna().values, color=color)
        ax.set_ylabel(label)
    plt.suptitle('STL Decomposition — WTI Crude Oil')
    plt.tight_layout(); plt.show()

## 5. Weather Data Integration

In [ ]:
# Try real weather data (US Gulf Coast) — falls back to simulated
weather = fetch_weather_data(29.7, -95.4, '2020-01-01', '2024-12-31')
if weather is None:
    weather = simulate_weather_data('2020-01-01', '2024-12-31')
    print('Using simulated weather data')
else:
    print(f'Fetched {len(weather)} weeks of weather data')
weather.tail()

## 6. Risk Score Analysis

In [ ]:
risk_df = compute_risk_scores(prices, 'WTI Crude Oil')
print(risk_df.to_string())